# Урок 09 - Патерн за метакогниция


## Настройка

Този бележник демонстрира шаблона за проектиране „Метакогниция“, използвайки Microsoft Agent Framework.

**Предварителни условия:**
- Разгръщане на Azure OpenAI, конфигурирано чрез променливи на средата
- Azure CLI е автентикиран (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Какво е метакогницията?

Метакогницията е **мислене за мисленето**. В контекста на AI агенти, това означава изграждане на агенти, които могат:

- **Саморефлектират** върху собствените си резултати и процес на разсъждение
- **Откриват грешки** и се възстановяват елегантно вместо да се провалят мълчаливо
- **Оценяват** дали техните отговори са пълни и полезни
- **Адаптират** своята стратегия, когато първоначалният подход не работи (например прибягване до резервна система)

Метакогнитивен агент не просто отговаря на въпроси — той наблюдава собственото си представяне и се приспособява в движение.


## Основни и резервни инструменти

Често срещан модел в метакогницията е **стратегията за резервен план**. Агентът първо опитва основен инструмент; ако той се провали (например грешка 404), агентът разпознава неуспеха и прозрачно преминава към резервен инструмент.

Това отразява системи в реалния свят, в които основните услуги може да са недостъпни и агентите трябва сами да диагностицират проблема, преди да изберат алтернативен път.

По-долу дефинираме два инструмента за търсене на полети:
- **Primary** — обхваща Париж, Токио и Барселона
- **Backup** — обхваща Берлин, Сидни и Ню Йорк


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Саморефлектиращ агент с възстановяване при грешки

Агентът по-долу е инструктиран първо да опита основната полетна система, да разпознава неизправности и прозрачно да превключва към резервната система. След всеки отговор той кратко самооценява дали е отговорил изцяло на въпроса на потребителя.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Шаблон за самооценка

Още един аспект на метакогницията е **самооценката**: отделен агент (или същият агент при втори преглед) преглежда отговор за пълнота, точност и полезност.

По-долу създаваме агент `ResponseEvaluator`, който оценява отговорите на агент за пътувания по три измерения.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Резюме

В този урок научихте как да изграждате **метакогнитивни агенти** с помощта на Microsoft Agent Framework:

- **Саморефлексия**: Агенти, които наблюдават собственото си разсъждаване и прозрачно комуникират какво се е случило.
- **Възстановяване при грешки с резервни опции**: Шаблон с основен и резервен инструмент, при който агентът открива неизправности (например 404 грешки) и автоматично опитва алтернативен източник.
- **Самооценка**: Отделен оценителен агент, който оценява отговорите по пълнота, точност и полезност.

Тези модели правят агентите по-устойчиви, прозрачни и надеждни — критични качества за внедряване в продукция.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Отказ от отговорност:
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, имайте предвид, че автоматичните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Не носим отговорност за каквито и да е недоразумения или погрешни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
